# 📧 Spam Email & SMS Classification - Model Training & Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**Case Study:** Spam Email / SMS Detection  
**Dataset Source:** [Kaggle SMS/Email Spam Collection Dataset](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)  
**Objective:** Train and test a text classification pipeline (TF-IDF + Logistic Regression), evaluate performance metrics, and serialize the complete pipeline into `model.joblib` for deployment via FastAPI and Docker.

## 1. Import Required Libraries
We import pandas, scikit-learn components for text processing and modeling, and joblib for serialization.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

print('Libraries loaded successfully!')

Libraries loaded successfully!


## 2. Load Dataset & Handle Encoding
Load the Kaggle dataset (`spam.csv` or `email_spam.csv`), handling potential encoding variants (latin-1 / utf-8).

In [2]:
# In Google Colab, upload the dataset or load from URL
url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'
try:
    df = pd.read_csv('email_spam.csv', encoding='utf-8')
except Exception:
    try:
        df = pd.read_csv('spam.csv', encoding='latin-1')
    except Exception:
        df = pd.read_csv(url, sep='\t', names=['label', 'text'])

# Clean and standardize columns
df = df.dropna(how='any', axis=1)
if len(df.columns) >= 2:
    df.columns = ['label', 'text'] + list(df.columns[2:])

print(f'Dataset Shape: {df.shape}')
print('Class Distribution:')
print(df['label'].value_counts())

Dataset Shape: (5572, 2)
Class Distribution:
ham     4825
spam     747
Name: count, dtype: int64


## 3. Preprocessing & Train-Test Split
Convert text labels to binary integers (ham: 0, spam: 1) and split into 80% training and 20% test sets using stratified sampling.

In [3]:
df['label_bin'] = df['label'].map({'ham': 0, 'spam': 1})
if df['label_bin'].isnull().any():
    df['label_bin'] = df['label'].astype('category').cat.codes

X = df['text']
y = df['label_bin']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training samples: {len(X_train)}')
print(f'Testing samples: {len(X_test)}')


Training samples: 4457
Testing samples: 1115


## 4. Build & Train Scikit-Learn Pipeline
Using a unified `Pipeline` ensures that TF-IDF text vectorization and classification are packaged together, preventing train-serve skew.

In [4]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        stop_words='english'
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        penalty='l2',
        random_state=42
    ))
])

pipeline.fit(X_train, y_train)
print('Pipeline training completed successfully!')

Pipeline training completed successfully!


## 5. Model Evaluation
Evaluate the model on unseen test data using Accuracy, Precision, Recall, F1-Score, and Confusion Matrix.

In [5]:
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {acc:.4f}\n')
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

Test Accuracy: 0.9785

Classification Report:
              precision    recall  f1-score   support

         Ham       0.98      1.00      0.99       966
        Spam       0.99      0.85      0.91       149

    accuracy                           0.98      1115
   macro avg       0.98      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115



## 6. Save Model and Parameters (`model.joblib`)
Fulfills Requirement 3: Serialize the trained pipeline into `model.joblib`.

In [6]:
joblib.dump(pipeline, 'model.joblib')
print('Successfully saved model to model.joblib')

Successfully saved model to model.joblib


## 7. Verify Model Reloading & Test Predictions
Verify that the saved model file loads correctly and produces accurate predictions.

In [7]:
loaded_model = joblib.load('model.joblib')
sample_texts = [
    'Claim your free prize now!',
    'Hello, can we schedule a team meeting tomorrow at 10 AM?'
]
print('Reloaded model test:')
for text in sample_texts:
    pred = loaded_model.predict([text])[0]
    prob = loaded_model.predict_proba([text])[0][pred]
    label = 'spam' if pred == 1 else 'ham'
    print(f'Input: {text} -> Predicted: {label} (Confidence: {prob*100:.2f}%)')


Reloaded model test:
Input: Claim your free prize now! -> Predicted: spam (Confidence: 83.83%)
Input: Hello, can we schedule a team meeting tomorrow at 10 AM? -> Predicted: ham (Confidence: 89.08%)
